# Stellar Spectral Classification with a Multi-Layer Perceptron

**Objective.** Build a neural network that classifies stars into
Morgan-Keenan (MK) spectral types -- O, B, A, F, G, K, M -- from their
optical spectra.

**What you will learn:**

| Topic | Section |
|---|---|
| Blackbody radiation and Planck's law | 1 |
| Atomic absorption lines in stellar atmospheres | 2 |
| The Morgan-Keenan classification system | 2 |
| Generating and exploring a synthetic spectral dataset | 3 |
| Data preprocessing for neural networks | 4 |
| Building and training an MLP in PyTorch | 5-6 |
| Evaluation: confusion matrix, per-class metrics | 7 |
| Gradient-based feature importance (which wavelengths matter?) | 8 |

**Prerequisites:** basic Python, introductory optics, no prior ML experience
required.

In [ ]:
# ── Install & import ─────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import warnings, random, os

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {device}")

---
## 1. Blackbody Radiation and Planck's Law

A star's continuous spectrum is well approximated by a **blackbody** at its
effective temperature $T_{\mathrm{eff}}$.  Planck's law gives the spectral
radiance:

$$
B_\lambda(\lambda, T) \;=\; \frac{2hc^2}{\lambda^5}
\;\frac{1}{\exp\!\bigl(\tfrac{hc}{\lambda k_B T}\bigr) - 1}
$$

where

| Symbol | Meaning | Value |
|--------|---------|-------|
| $h$ | Planck constant | $6.626 \times 10^{-34}\;\mathrm{J\,s}$ |
| $c$ | Speed of light | $2.998 \times 10^{8}\;\mathrm{m/s}$ |
| $k_B$ | Boltzmann constant | $1.381 \times 10^{-23}\;\mathrm{J/K}$ |

**Wien's displacement law** tells us the peak wavelength shifts with
temperature: $\lambda_{\max} = b / T$, with $b \approx 2.898 \times 10^{-3}\;\mathrm{m\,K}$.

- Hot O-type stars ($T \sim 40\,000\;\mathrm{K}$): peak in the UV, appear
  blue-white.
- Cool M-type stars ($T \sim 3\,000\;\mathrm{K}$): peak in the infrared,
  appear red.

The cell below plots Planck curves for several temperatures.

In [ ]:
# ── Physical constants ───────────────────────────────────────────────────
h  = 6.62607015e-34   # J s
c  = 2.99792458e8     # m/s
kB = 1.380649e-23     # J/K

def planck(lam, T):
    '''Spectral radiance B_lambda(lambda, T) in W / (m^3 sr).

    Parameters
    ----------
    lam : array-like   Wavelength in metres.
    T   : float        Temperature in Kelvin.
    '''
    lam = np.asarray(lam, dtype=np.float64)
    a = 2.0 * h * c**2 / lam**5
    b = np.expm1(h * c / (lam * kB * T))     # exp(...) - 1, numerically stable
    return a / b

# Plot for representative MK temperatures
temperatures = {
    "O  (40 000 K)": 40000,
    "B  (20 000 K)": 20000,
    "A  (9 000 K)":  9000,
    "F  (7 000 K)":  7000,
    "G  (5 800 K)":  5800,
    "K  (4 500 K)":  4500,
    "M  (3 200 K)":  3200,
}

lam_m = np.linspace(100e-9, 2500e-9, 2000)      # 100 nm to 2500 nm
lam_nm = lam_m * 1e9

fig, ax = plt.subplots(figsize=(10, 5))
cmap = plt.cm.RdYlBu_r
for i, (label, T) in enumerate(temperatures.items()):
    B = planck(lam_m, T)
    B_norm = B / B.max()                         # normalise for visibility
    ax.plot(lam_nm, B_norm, label=label,
            color=cmap(i / (len(temperatures) - 1)), lw=1.8)

ax.axvspan(380, 700, alpha=0.08, color="green", label="visible band")
ax.set_xlabel("Wavelength (nm)"); ax.set_ylabel("Normalised spectral radiance")
ax.set_title("Planck curves for different stellar temperatures")
ax.legend(fontsize=8); ax.set_xlim(100, 2500); ax.set_ylim(0, 1.05)
plt.tight_layout(); plt.show()

---
## 2. Absorption Lines and the Morgan-Keenan System

A stellar atmosphere absorbs photons at specific wavelengths determined by
the atomic and molecular species present and their excitation states.  The
**depth** of each absorption line depends on temperature and surface
gravity.

### Key spectral features

| Feature | Wavelength (nm) | Strong in types |
|---------|---------------:|-----------------|
| Ca II K | 393.4 | F, G, K |
| Ca II H | 396.8 | F, G, K |
| H-$\delta$ | 410.2 | A, B |
| H-$\gamma$ | 434.0 | A, B |
| H-$\beta$ | 486.1 | A, B (strongest in A) |
| Mg I b | 518.4 | G, K |
| Fe I | 527.0 | G, K |
| Na D | 589.3 | K, M |
| He I | 587.6 | O, B |
| H-$\alpha$ | 656.3 | A (max), weaker in B, F |
| TiO band | 705-720 | M (dominant) |

### The MK classification

The **Morgan-Keenan** system assigns each star a **spectral type**
(O, B, A, F, G, K, M) based on these features.  The sequence runs from
hottest (O) to coolest (M), remembered by the mnemonic:

> *Oh Be A Fine Girl/Guy, Kiss Me*

Each type is subdivided 0-9 (e.g., G2 for the Sun), but in this exercise
we classify into the **seven main types** only.

The next cells define a physics-based spectrum generator that creates
realistic synthetic spectra.

---
## 3. Building a Synthetic Stellar Spectrum Dataset

We generate spectra by:

1. Drawing a random effective temperature from the range for each spectral
   type.
2. Computing the Planck continuum.
3. Imprinting absorption lines as **Gaussian dips** whose depths depend on
   spectral type and temperature.
4. Adding photon-counting noise (Poisson-like) and detector read noise
   (Gaussian).

This gives us full control over the dataset and ensures every student
gets the same results for a given random seed.

In [ ]:
# ── Spectral type definitions ────────────────────────────────────────────

# Temperature ranges (K) for each MK type
TYPE_TEMPS = {
    "O": (30000, 50000),
    "B": (10000, 30000),
    "A": (7500,  10000),
    "F": (6000,   7500),
    "G": (5200,   6000),
    "K": (3700,   5200),
    "M": (2400,   3700),
}

CLASS_NAMES = list(TYPE_TEMPS.keys())       # O, B, A, F, G, K, M
NUM_CLASSES = len(CLASS_NAMES)

# Wavelength grid: 380-900 nm at 0.5 nm resolution -> 1040 features
LAMBDA_MIN, LAMBDA_MAX, DLAMBDA = 380.0, 899.5, 0.5
WAVELENGTHS = np.arange(LAMBDA_MIN, LAMBDA_MAX + DLAMBDA, DLAMBDA)  # nm
NUM_FEATURES = len(WAVELENGTHS)
print(f"Wavelength grid: {NUM_FEATURES} points from {LAMBDA_MIN} to "
      f"{WAVELENGTHS[-1]} nm (step {DLAMBDA} nm)")

In [ ]:
# ── Absorption-line profiles ─────────────────────────────────────────────
#
# Each line is (centre_nm, width_nm, {type: depth}).
# Depth 0 = no line; depth 1 = very strong absorption.

LINES = [
    # Calcium II  K & H
    (393.4, 1.0, {"F": 0.30, "G": 0.50, "K": 0.60, "M": 0.35}),
    (396.8, 0.9, {"F": 0.28, "G": 0.48, "K": 0.58, "M": 0.33}),
    # Hydrogen Balmer series
    (410.2, 1.2, {"O": 0.15, "B": 0.40, "A": 0.55, "F": 0.25}),        # H-delta
    (434.0, 1.3, {"O": 0.18, "B": 0.45, "A": 0.60, "F": 0.28}),        # H-gamma
    (486.1, 1.5, {"O": 0.20, "B": 0.50, "A": 0.70, "F": 0.30, "G": 0.10}),  # H-beta
    (656.3, 1.8, {"O": 0.10, "B": 0.35, "A": 0.65, "F": 0.25, "G": 0.08}),  # H-alpha
    # Helium I  (hot stars)
    (587.6, 1.0, {"O": 0.45, "B": 0.30}),
    # Helium II  (very hot)
    (468.6, 0.8, {"O": 0.50}),
    # Sodium D
    (589.3, 0.8, {"G": 0.15, "K": 0.40, "M": 0.50}),
    # Magnesium I b
    (518.4, 1.0, {"G": 0.25, "K": 0.35, "M": 0.20}),
    # Iron I
    (527.0, 0.7, {"G": 0.20, "K": 0.30, "M": 0.15}),
    (532.8, 0.6, {"G": 0.15, "K": 0.25, "M": 0.12}),
    # TiO molecular bands (cool stars) -- modelled as broad features
    (705.0, 8.0, {"K": 0.08, "M": 0.55}),
    (715.0, 6.0, {"M": 0.50}),
    (760.0, 10.0, {"M": 0.40}),
    (845.0, 12.0, {"M": 0.35}),
]


def generate_spectrum(spectral_type, rng=None):
    '''Generate one synthetic stellar spectrum.

    Returns
    -------
    flux : ndarray of shape (NUM_FEATURES,)
        Normalised flux with absorption lines and noise.
    '''
    if rng is None:
        rng = np.random.default_rng()

    # 1. Random temperature within the type's range
    lo, hi = TYPE_TEMPS[spectral_type]
    T = rng.uniform(lo, hi)

    # 2. Planck continuum (in nm, then normalise to [0, 1])
    lam_m = WAVELENGTHS * 1e-9
    continuum = planck(lam_m, T)
    continuum = continuum / continuum.max()

    # 3. Imprint absorption lines
    flux = continuum.copy()
    for centre, width, depths in LINES:
        depth = depths.get(spectral_type, 0.0)
        if depth > 0:
            # Add small random variation to depth and width
            d = depth * rng.uniform(0.8, 1.2)
            w = width * rng.uniform(0.9, 1.1)
            profile = d * np.exp(-0.5 * ((WAVELENGTHS - centre) / w) ** 2)
            flux *= (1.0 - profile)

    # 4. Add noise: signal-dependent (Poisson-like) + read noise
    snr = rng.uniform(30, 100)                        # signal-to-noise ratio
    noise = flux / snr * rng.standard_normal(NUM_FEATURES)
    read_noise = 0.005 * rng.standard_normal(NUM_FEATURES)
    flux = flux + noise + read_noise
    flux = np.clip(flux, 0, None)

    return flux.astype(np.float32)


# Quick test
test_flux = generate_spectrum("A", rng=np.random.default_rng(0))
print(f"Sample A-type spectrum: shape={test_flux.shape}, "
      f"min={test_flux.min():.4f}, max={test_flux.max():.4f}")

In [ ]:
# ── Generate the full dataset ────────────────────────────────────────────

N_PER_CLASS = 500          # 500 spectra per type -> 3500 total
SAMPLES_TOTAL = N_PER_CLASS * NUM_CLASSES

rng = np.random.default_rng(SEED)
X_all = np.empty((SAMPLES_TOTAL, NUM_FEATURES), dtype=np.float32)
y_all = np.empty(SAMPLES_TOTAL, dtype=np.int64)

for cls_idx, cls_name in enumerate(CLASS_NAMES):
    for j in range(N_PER_CLASS):
        idx = cls_idx * N_PER_CLASS + j
        X_all[idx] = generate_spectrum(cls_name, rng=rng)
        y_all[idx] = cls_idx

print(f"Dataset: {X_all.shape[0]} spectra, {X_all.shape[1]} wavelength bins, "
      f"{NUM_CLASSES} classes")
print("Class distribution:", {CLASS_NAMES[i]: (y_all == i).sum() for i in range(NUM_CLASSES)})

### 3.1 Visualising the spectra

Let's plot a few example spectra from each spectral type.  Notice:

- The **continuum slope** shifts from blue-peaked (O) to red-peaked (M) --
  this is Wien's law in action.
- **Hydrogen Balmer lines** (H-$\alpha$, H-$\beta$, H-$\gamma$, H-$\delta$)
  are deepest in A-type stars.
- **Calcium and metallic lines** grow stronger in F-G-K.
- **TiO molecular bands** (broad dips around 700-850 nm) appear only in
  M-type stars.

In [ ]:
# ── Plot example spectra ─────────────────────────────────────────────────
fig, axes = plt.subplots(NUM_CLASSES, 1, figsize=(12, 14), sharex=True)

colours = ["#4400cc", "#0066ff", "#00aadd", "#44bb44",
           "#ddaa00", "#ee6600", "#cc0000"]

for cls_idx, (ax, name, col) in enumerate(zip(axes, CLASS_NAMES, colours)):
    # Plot 5 random spectra from this class
    mask = y_all == cls_idx
    idxs = np.where(mask)[0][:5]
    for i in idxs:
        ax.plot(WAVELENGTHS, X_all[i], alpha=0.6, lw=0.7, color=col)
    ax.set_ylabel(f"Type {name}", fontsize=11, fontweight="bold")
    ax.set_ylim(-0.05, 1.15)
    ax.tick_params(labelsize=8)

    # Mark key lines
    for lam, label in [(393.4, "CaK"), (486.1, r"H$\beta$"),
                        (587.6, "HeI"), (656.3, r"H$\alpha$"),
                        (589.3, "NaD"), (710, "TiO")]:
        ax.axvline(lam, color="grey", lw=0.4, ls="--", alpha=0.5)

axes[-1].set_xlabel("Wavelength (nm)", fontsize=11)
axes[0].set_title("Synthetic stellar spectra by MK type", fontsize=13)
plt.tight_layout(); plt.show()

---
## 4. Data Preprocessing

Before feeding spectra into a neural network we apply two standard steps:

1. **Continuum normalisation.**  Divide each spectrum by a smooth fit to its
   continuum so that the network focuses on absorption features rather than
   the overall Planck slope.  For simplicity we normalise each spectrum to
   have zero mean and unit variance (standardisation).

2. **Train / validation / test split.**  We use 70% / 15% / 15%.

> **Why standardise?** Neural networks converge faster when input features
> have similar scales.  Without standardisation the network would waste
> capacity learning the Planck slope, which is already encoded in the class
> label.

In [ ]:
# ── Standardise: per-spectrum zero mean, unit variance ───────────────────
X_mean = X_all.mean(axis=1, keepdims=True)
X_std  = X_all.std(axis=1, keepdims=True) + 1e-8
X_norm = (X_all - X_mean) / X_std

# Convert to tensors
X_tensor = torch.tensor(X_norm)
y_tensor = torch.tensor(y_all)

# Split: 70 / 15 / 15
dataset = TensorDataset(X_tensor, y_tensor)
n_total = len(dataset)
n_train = int(0.70 * n_total)
n_val   = int(0.15 * n_total)
n_test  = n_total - n_train - n_val

train_ds, val_ds, test_ds = random_split(
    dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(SEED)
)

BATCH = 64
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH)
test_dl  = DataLoader(test_ds,  batch_size=BATCH)

print(f"train {n_train} | val {n_val} | test {n_test}")
xb, yb = next(iter(train_dl))
print(f"Batch shapes: X {xb.shape}, y {yb.shape}")

---
## 5. The Multi-Layer Perceptron (MLP)

An MLP is a sequence of **fully-connected (linear) layers** separated by
non-linear **activation functions**.  For an input vector $\mathbf{x} \in
\mathbb{R}^d$:

$$
\mathbf{h}_1 = \sigma(W_1\,\mathbf{x} + \mathbf{b}_1)
$$
$$
\mathbf{h}_2 = \sigma(W_2\,\mathbf{h}_1 + \mathbf{b}_2)
$$
$$
\hat{\mathbf{y}} = W_3\,\mathbf{h}_2 + \mathbf{b}_3
$$

where $\sigma$ is the activation function (we use **ReLU**: $\sigma(z) =
\max(0, z)$).  The output $\hat{\mathbf{y}} \in \mathbb{R}^C$ contains
raw **logits** for each of the $C = 7$ classes.

### Why not a CNN?

Our input is a 1-D spectrum on a **fixed, uniform wavelength grid**.
Absorption lines have known, fixed positions.  An MLP can learn to
directly read the flux at those positions.  A CNN would be preferable if
the lines could shift (e.g., due to radial velocity / redshift), but for
this introductory exercise an MLP is sufficient and more transparent.

### Architecture choices

| Hyperparameter | Value | Rationale |
|---|---|---|
| Hidden layers | 3 | Enough capacity for 7-class problem |
| Hidden sizes | 256, 128, 64 | Gradual compression |
| Activation | ReLU | Simple, effective |
| Dropout | 0.3 | Regularisation |
| Output | 7 (softmax via CrossEntropyLoss) | 7 MK types |

In [ ]:
# ── MLP definition ───────────────────────────────────────────────────────

class StellarMLP(nn.Module):
    def __init__(self, n_features, n_classes, hidden=(256, 128, 64), drop=0.3):
        super().__init__()
        layers = []
        prev = n_features
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h),
                       nn.ReLU(), nn.Dropout(drop)]
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

model = StellarMLP(NUM_FEATURES, NUM_CLASSES).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params:,}")

---
## 6. Training the MLP

We use:

- **Cross-entropy loss:**
  $\mathcal{L} = -\sum_{c=1}^{C} y_c \log \hat{p}_c$
  where $\hat{p}_c = \mathrm{softmax}(\hat{y})_c$. This is the standard
  loss for multi-class classification.

- **Adam optimiser** with learning rate $10^{-3}$.

- **Learning rate scheduler:** reduce on plateau (halve LR if validation
  loss stalls for 5 epochs).

The training loop below tracks both loss and accuracy on the training and
validation sets.

In [ ]:
# ── Training setup ───────────────────────────────────────────────────────
EPOCHS = 60
LR = 1e-3

criterion = nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimiser, mode="min", factor=0.5, patience=5
)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

def run_epoch(loader, train=True):
    model.train(train)
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            if train:
                optimiser.zero_grad()
                loss.backward()
                optimiser.step()
            total_loss += loss.item() * len(yb)
            correct += (logits.argmax(1) == yb).sum().item()
            total += len(yb)
    return total_loss / total, correct / total

for epoch in range(1, EPOCHS + 1):
    tl, ta = run_epoch(train_dl, train=True)
    vl, va = run_epoch(val_dl, train=False)
    scheduler.step(vl)
    history["train_loss"].append(tl)
    history["val_loss"].append(vl)
    history["train_acc"].append(ta)
    history["val_acc"].append(va)
    if epoch % 10 == 0 or epoch == 1:
        lr_now = optimiser.param_groups[0]["lr"]
        print(f"epoch {epoch:3d} | train loss {tl:.4f} acc {ta:.3f} | "
              f"val loss {vl:.4f} acc {va:.3f} | lr {lr_now:.1e}")

print(f"\nBest val accuracy: {max(history['val_acc']):.3f} "
      f"(epoch {np.argmax(history['val_acc']) + 1})")

In [ ]:
# ── Training curves ──────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history["train_loss"], label="train"); ax1.plot(history["val_loss"], label="val")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.legend(); ax1.set_title("Loss")

ax2.plot(history["train_acc"],  label="train"); ax2.plot(history["val_acc"],  label="val")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy"); ax2.legend(); ax2.set_title("Accuracy")

plt.tight_layout(); plt.show()

---
## 7. Evaluation on the Test Set

We compute:

- **Overall accuracy** -- fraction of correctly classified spectra.
- **Confusion matrix** -- which types get confused with which.
- **Per-class precision, recall, F1** -- important when classes are not
  equally easy.

In [ ]:
# ── Test evaluation ──────────────────────────────────────────────────────
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in test_dl:
        xb = xb.to(device)
        preds = model(xb).argmax(1).cpu()
        all_preds.append(preds)
        all_labels.append(yb)

all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()
test_acc = (all_preds == all_labels).mean()
print(f"TEST accuracy: {test_acc:.4f}  ({(all_preds != all_labels).sum()} "
      f"mistakes out of {len(all_labels)})\n")

print(classification_report(all_labels, all_preds,
                            target_names=CLASS_NAMES, digits=3))

In [ ]:
# ── Confusion matrix ─────────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASS_NAMES)
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion Matrix")

for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        val = cm[i, j]
        colour = "white" if val > cm.max() / 2 else "black"
        ax.text(j, i, str(val), ha="center", va="center", color=colour, fontsize=10)

plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.show()

### Interpreting the confusion matrix

Look at which types are most often confused:

- **B and O** may overlap because both have helium lines and hydrogen
  lines; the main difference is continuum slope and He II strength.
- **F and G** can be similar -- both have moderate hydrogen lines and
  growing metallic features.
- **M stars** should be very distinct because of the strong TiO bands.

> **Question for students:** Which pair of classes is hardest to
> distinguish?  What physical reason explains this?

---
## 8. Which Wavelengths Does the Network Use?

We can compute **gradient-based feature importance**: for each input
wavelength bin, how much does a small change in flux affect the predicted
class probability?

$$
\text{importance}_j = \left| \frac{\partial \hat{p}_c}{\partial x_j} \right|
$$

averaged over many spectra.  Peaks in the importance curve should align
with known absorption lines.

In [ ]:
# ── Gradient-based feature importance ────────────────────────────────────
model.eval()
importances = np.zeros(NUM_FEATURES, dtype=np.float64)
n_samples = 0

for xb, yb in test_dl:
    xb = xb.to(device).requires_grad_(True)
    logits = model(xb)
    probs = F.softmax(logits, dim=1)
    # Gather the probability of the true class
    true_probs = probs.gather(1, yb.to(device).unsqueeze(1)).sum()
    true_probs.backward()
    importances += xb.grad.abs().sum(dim=0).cpu().numpy()
    n_samples += len(yb)

importances /= n_samples

# Plot
fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(WAVELENGTHS, importances, alpha=0.4, color="steelblue")
ax.plot(WAVELENGTHS, importances, lw=0.8, color="steelblue")

# Annotate known lines
line_labels = {393.4: "Ca II K", 486.1: r"H$\beta$", 587.6: "He I",
               589.3: "Na D", 656.3: r"H$\alpha$", 710: "TiO"}
for lam, label in line_labels.items():
    ax.axvline(lam, color="red", lw=0.6, ls="--", alpha=0.6)
    ax.text(lam + 3, importances.max() * 0.9, label,
            fontsize=7, rotation=90, color="red")

ax.set_xlabel("Wavelength (nm)"); ax.set_ylabel("Mean |gradient|")
ax.set_title("Feature importance: which wavelengths drive the classification?")
plt.tight_layout(); plt.show()

> **Discussion:** The importance peaks should cluster around the known
> absorption lines.  The network has effectively "rediscovered" which
> spectral features matter for classification -- the same features that
> astronomers have used since Annie Jump Cannon's pioneering work on the
> Harvard classification in the early 1900s.

---
## 9. Exercises

**Exercise 1 -- Architecture search.**
Try different hidden layer sizes (e.g., `[512, 256]` or `[128, 64, 32, 16]`).
How does the number of parameters affect test accuracy?  Is there a point
of diminishing returns?

**Exercise 2 -- Noise robustness.**
Regenerate the dataset with lower SNR values (e.g., `snr = rng.uniform(10, 30)`)
simulating fainter stars.  How does classification accuracy degrade?  Which
spectral types are most affected?

**Exercise 3 -- Dimensionality reduction.**
Apply PCA to the spectra before training.  How many principal components
are needed to retain 99% of the variance?  Train the MLP on the PCA
features -- does accuracy change?  Is training faster?

**Exercise 4 -- Radial velocity shift.**
Modify `generate_spectrum` to apply a random Doppler shift:
$\lambda' = \lambda\,(1 + v/c)$ with $v$ drawn from
$\mathcal{N}(0, 200\;\mathrm{km/s})$.  Does the MLP still work?  Would a
CNN be more appropriate here?  Why?

**Exercise 5 -- Real data.**
Download spectra from the [SDSS SkyServer](https://skyserver.sdss.org/) or
[LAMOST DR9](http://www.lamost.org/dr9/).  Preprocess them onto the same
wavelength grid and test the trained model.  What domain gap do you observe?

In [ ]:
# ── Starter code for Exercise 3 (PCA) ───────────────────────────────────
from sklearn.decomposition import PCA

# Fit PCA on the training split only
train_indices = train_ds.indices
X_train_np = X_norm[train_indices]

pca = PCA(n_components=50)
pca.fit(X_train_np)

cumvar = np.cumsum(pca.explained_variance_ratio_)
n99 = np.searchsorted(cumvar, 0.99) + 1
print(f"Components for 99% variance: {n99}")

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(range(1, 51), cumvar, "o-", ms=3)
ax.axhline(0.99, color="red", ls="--", label="99% threshold")
ax.set_xlabel("Number of components"); ax.set_ylabel("Cumulative explained variance")
ax.legend(); plt.tight_layout(); plt.show()

# TODO: transform the data with pca.transform(), rebuild DataLoaders,
# and retrain the MLP with n_features = n99.

---

*Notebook prepared for optics / astrophysics students learning machine
learning.  The synthetic spectra are physically motivated but simplified;
real stellar classification uses much higher resolution spectra and
accounts for radial velocity, interstellar reddening, and metallicity.*